In [ ]:
# ============================================================
# Swissdox@LiRI → API → TSV.xz → DataFrame nettoyé (VS Code)
# ============================================================

import os
import time
from io import BytesIO
import re
import html
import requests
import pandas as pd
import yaml
from dotenv import load_dotenv

# -------------------------------
# 0. ENV + API KEYS
# -------------------------------

# Charge les variables d'environnement depuis .env (à créer dans ton projet)
load_dotenv()

API_KEY = os.getenv("SWISSDOX_API_KEY")
API_SECRET = os.getenv("SWISSDOX_API_SECRET")

if not API_KEY or not API_SECRET:
    raise RuntimeError("Swissdox API keys missing. Set SWISSDOX_API_KEY and SWISSDOX_API_SECRET in your .env file.")

API_BASE_URL   = "https://swissdox.linguistik.uzh.ch/api"
API_URL_QUERY  = f"{API_BASE_URL}/query"
API_URL_STATUS = f"{API_BASE_URL}/status"

HEADERS = {
    "X-API-Key": API_KEY,
    "X-API-Secret": API_SECRET,
}

from datetime import datetime

QUERY_BASE_NAME = "BuerokratieVerwaltung_2025"
QUERY_NAME = f"{QUERY_BASE_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
QUERY_COMMENT   = "Requête générée depuis VS Code"
EXPIRATION_DATE = "2025-12-31"   # ou "" si tu t'en fiches

# Période
START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"

# Langues
LANGUAGES = ["de", "fr"]

# Journaux / sources
SOURCES = [
    "NZZO",
    "NNTA",
    "NNHEU",
    "ZWSO",
    "TPS",
    "NZZ",
    "TA",
    "ZWAO",
    "TPSO",
    "HEU",
    "ZWAS",
    "NZZS",
    "ZWAI",
]

# Résultats max
MAX_RESULTS = 10000

# -------------------------------
# 1. FONCTIONS DE NETTOYAGE TEXTE
# -------------------------------

def clean_text(text: str) -> str:
    """Nettoie un champ texte pour usage dans pandas."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip(' "“”„\'')
    return text.strip()

def clean_xml_swissdox(text: str) -> str:
    """Supprime les balises XML Swissdox et normalise le texte."""
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = re.sub(r"</p>", "\n", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# -------------------------------
# 2. CONSTRUCTION DU YAML
# -------------------------------

query_block = {
    "sources": SOURCES,
    "dates": [
        {
            "from": START_DATE,
            "to": END_DATE,
        }
    ],
    "languages": LANGUAGES,
}

KEYWORDS = ["Bürokratie", "bureaucratie"]

query_block["content"] = {
    "OR": [
        {
            "OR": KEYWORDS
        },
        {
            "AND": [
                "öffentliche",
                "Verwaltung",
                {
                    "OR": [
                        "Bund*",
                        "bundes*",
                        "Kanton*",
                        "kantonal*"
                    ]
                }
            ]
        },
        {
            "AND": [
                "administration",
                "publique",
                {
                    "OR": [
                        "fédéral*",
                        "federal*",
                        "federale*",
                        "cantonal*",
                        "cantonale*"
                    ]
                }
            ]
        }
    ]
}

yaml_payload = {
    "query": query_block,
    "result": {
        "format": "TSV",
        "maxResults": MAX_RESULTS,
        "columns": [
            "id",
            "pubtime",
            "medium_code",
            "medium_name",
            "rubric",
            "regional",
            "doctype",
            "doctype_description",
            "language",
            "char_count",
            "dateline",
            "head",
            "subhead",
            "content_id",
            "content",
        ],
    },
    "version": "1.2",
}

yaml_query = yaml.safe_dump(
    yaml_payload,
    sort_keys=False,
    allow_unicode=True,
)

print("===== YAML envoyé à Swissdox =====")
print(yaml_query)

# -------------------------------
# 3. ENVOI DE LA REQUÊTE /query
# -------------------------------

data = {
    "query": yaml_query,
    "name": QUERY_NAME,
    "comment": QUERY_COMMENT,
    "expirationDate": EXPIRATION_DATE,
}

r = requests.post(API_URL_QUERY, headers=HEADERS, data=data)

print("===== Réponse /query =====")
print("Status code :", r.status_code)
print("Texte :", r.text)

r.raise_for_status()
resp_json = r.json()
if resp_json.get("result") != "ok":
    raise SystemExit(f"❌ Swissdox renvoie un résultat non-ok : {resp_json}")

query_id = resp_json.get("queryId") or resp_json.get("id")
if not query_id:
    raise SystemExit(f"❌ Impossible de récupérer queryId dans la réponse : {resp_json}")

print(f"✅ Requête soumise avec succès. queryId = {query_id}")

# -------------------------------
# 4. RÉCUPÉRATION DU DOWNLOAD_URL VIA /status
# -------------------------------

print("\n⏳ Récupération du downloadUrl via /status...")

download_url = None
job_info = None

for i in range(300):
    rs = requests.get(API_URL_STATUS, headers=HEADERS)
    rs.raise_for_status()

    status_list = rs.json()
    job_info = next((job for job in status_list if job.get("id") == query_id), None)

    print("----- Statut actuel -----")
    print(job_info)

    if job_info is None:
        print("ℹ️ Job pas encore visible dans /status, nouvelle tentative...")
        time.sleep(5)
        continue

    status = job_info.get("status")
    actual = job_info.get("actualResults")
    err    = job_info.get("error")
    download_url = job_info.get("downloadUrl")

    if err:
        raise SystemExit(f"❌ Erreur Swissdox pour cette requête : {err}")

    if download_url:
        print("✅ URL de téléchargement trouvée :", download_url)
        break

    if status == "finished" and (actual == 0 or actual is None) and not download_url:
        print("ℹ️ Requête terminée mais aucun résultat (actualResults = 0).")
        break

    time.sleep(5)

if not download_url:
    raise SystemExit("❌ Aucun fichier à télécharger (downloadUrl manquant).")

# -------------------------------
# 5. TÉLÉCHARGEMENT + LECTURE TSV.XZ DIRECTE
# -------------------------------

if download_url.startswith("http"):
    download_full_url = download_url
elif download_url.startswith("/"):
    download_full_url = f"{API_BASE_URL}{download_url}"
else:
    download_full_url = f"{API_BASE_URL}/download/{download_url}"

print("\n🔻 Téléchargement depuis :", download_full_url)

r_dl = requests.get(download_full_url, headers=HEADERS)
r_dl.raise_for_status()

print("Taille du fichier téléchargé : %.2f KB" % (len(r_dl.content) / 1024))

tsv_bytes = BytesIO(r_dl.content)

df = pd.read_csv(
    tsv_bytes,
    sep="\t",
    compression="xz"
)

# Conversion de pubtime en date (YYYY-MM-DD)
if "pubtime" in df.columns:
    df["pubtime"] = pd.to_datetime(df["pubtime"].astype(str), errors="coerce", utc=True).dt.date

print("Nombre de lignes chargées :", len(df))
print("Colonnes :", df.columns.tolist())

# -------------------------------
# 6. NETTOYAGE DES TEXTES
# -------------------------------

TEXT_COLS_TO_CLEAN = [
    "medium_name",
    "rubric",
    "dateline",
    "head",
    "subhead",
]

for col in TEXT_COLS_TO_CLEAN:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

if "content" in df.columns:
    df["content"] = df["content"].apply(clean_xml_swissdox)
    df["content"] = df["content"].apply(clean_text)
else:
    print("⚠️ Colonne 'content' absente du fichier téléchargé.")

# -------------------------------
# 7. APERÇU
# -------------------------------

df.head()

In [ ]:
# ============================================================
# 8. THEME PRINCIPAL PAR EMBEDDINGS + COSINE SIMILARITY
#    (basé sur head + subhead par défaut)
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np

# -------------------------------
# 8.1 Choix du texte à encoder
# -------------------------------
# Recommandé: head + subhead (rapide + souvent plus "thème principal" que content)
df["text_for_theme"] = (df["head"].fillna("") + " " + df["head"].fillna("") + " — " + df["subhead"].fillna("")).str.strip(" —")

# Fallback: si head/subhead vides, on prend le début du content (évite les textes trop longs)
if "content" in df.columns:
    mask_empty = df["text_for_theme"].eq("")
    df.loc[mask_empty, "text_for_theme"] = (
        df.loc[mask_empty, "content"].fillna("").astype(str).str.slice(0, 1200)
    )

# -------------------------------
# 8.2 Définis ta taxonomie (à adapter)
#     Astuce: label + petite description = mieux que label seul
# -------------------------------
THEMES = [

    ("Chancellerie fédérale (ChF/BK)",
     "FR: coordination du Conseil fédéral, communication gouvernementale, processus de décision; acteurs: Chancellerie fédérale, Conseil fédéral; "
     "instruments: messages, ordonnances, consultations, votations; "
     "vocabulaire: 'Conseil fédéral', 'Chancellerie', 'message', 'ordonnance', 'consultation', 'votation', 'communiqué'. "
     "DE: Koordination Bundesrat, Regierungskommunikation; Akteure: Bundeskanzlei, Bundesrat; "
     "Instrumente: Botschaft, Verordnung, Vernehmlassung, Abstimmung; "
     "Vokabular: 'Bundesrat', 'Bundeskanzlei', 'Botschaft', 'Verordnung', 'Vernehmlassung', 'Abstimmung'."),

    ("DFAE / EDA – Affaires étrangères",
     "FR: politique extérieure, diplomatie, coopération et aide humanitaire, droit international, affaires consulaires; acteurs: DFAE, ambassades, ONU/UE, DDC, directions (DDIP, consulaire); "
     "instruments: négociations, accords/traités, sanctions internationales, protection consulaire, programmes de coopération, aide humanitaire; "
     "vocabulaire: 'diplomatie', 'accord', 'traité', 'ONU', 'UE', 'DDC', 'aide humanitaire', 'coopération', 'consulat', 'passeport', 'rapatriement', 'droit international', 'CEDH'. "
     "DE: Aussenpolitik, Diplomatie, Entwicklung und humanitäre Hilfe, Völkerrecht, Konsularwesen; Akteure: EDA, Botschaften, UNO/EU, DEZA, Direktionen (Völkerrecht, Konsular); "
     "Instrumente: Verhandlungen, Abkommen/Verträge, internationale Sanktionen, konsularischer Schutz, Entwicklungsprogramme, humanitäre Hilfe; "
     "Vokabular: 'Diplomatie', 'Abkommen', 'Vertrag', 'UNO', 'EU', 'DEZA', 'humanitäre Hilfe', 'Zusammenarbeit', 'Konsulat', 'Pass', 'Repatriierung', 'Völkerrecht', 'EMRK'."),

    ("DFI / EDI – Intérieur (santé, social, culture, égalité, statistiques)",
     "FR: santé publique, assurances sociales, sécurité alimentaire/vétérinaire, culture, égalité, statistiques et archives; acteurs: DFI, OFSP, OFAS, OSAV, OFC, BFEG, OFS, Archives fédérales, MétéoSuisse; "
     "instruments: recommandations/ordonnances sanitaires, surveillance épidémiologique, réglementation LAMal, prestations AVS/AI, contrôles alimentaires, subventions culturelles, politiques égalité, indicateurs/statistiques, archivage, alertes météo; "
     "vocabulaire: 'primes', 'LAMal', 'hôpital', 'vaccination', 'médicaments', 'AVS', 'AI', 'rentes', 'prestations', 'sécurité alimentaire', 'rappel de produit', 'zoonose', 'culture', 'subventions', 'égalité salariale', 'statistique', 'archives', 'alerte météo', 'climat'. "
     "DE: Gesundheit, Sozialversicherungen, Lebensmittel-/Veterinärwesen, Kultur, Gleichstellung, Statistik/Archive; Akteure: EDI, BAG, BSV, BLV, BAK, EBG, BFS, Bundesarchiv, MeteoSchweiz; "
     "Instrumente: Empfehlungen/Verordnungen, epidemiologische Überwachung, KVG, AHV/IV-Leistungen, Lebensmittelkontrollen, Kulturförderung, Gleichstellungspolitik, Indikatoren/Statistik, Archivierung, Wetterwarnungen; "
     "Vokabular: 'Prämien', 'KVG', 'Spital', 'Impfung', 'Medikamente', 'AHV', 'IV', 'Renten', 'Leistungen', 'Lebensmittelsicherheit', 'Produktrückruf', 'Zoonose', 'Kultur', 'Förderung', 'Lohngleichheit', 'Statistik', 'Archiv', 'Wetterwarnung', 'Klima'."),

    ("DFJP / EJPD – Justice et police (asile, migration, sécurité, surveillance)",
     "FR: justice, police fédérale, poursuites, asile et migration, surveillance légale des communications; acteurs: DFJP, SEM, fedpol, OFJ, autorités pénales, SCPT; "
     "instruments: procédures d’asile, décisions/renvois, enquêtes, entraide, révisions légales, interceptions sur base légale; "
     "vocabulaire: 'asile', 'requérant', 'procédure', 'renvoi', 'centre fédéral', 'migration', 'criminalité', 'terrorisme', 'enquête', 'Interpol', 'code', 'révision', 'entraide', 'surveillance', 'télécommunications', 'interception'. "
     "DE: Justiz, Bundespolizei, Strafverfolgung, Asyl/Migration, gesetzliche Überwachung Kommunikation; Akteure: EJPD, SEM, fedpol, BJ, Strafbehörden, ÜPF; "
     "Instrumente: Asylverfahren, Entscheide/Wegweisung, Ermittlungen, Rechtshilfe, Gesetzesrevisionen, Überwachung mit Anordnung; "
     "Vokabular: 'Asyl', 'Asylsuchende', 'Verfahren', 'Wegweisung', 'Bundesasylzentrum', 'Migration', 'Kriminalität', 'Terrorismus', 'Ermittlung', 'Interpol', 'Gesetz', 'Revision', 'Rechtshilfe', 'Überwachung', 'Fernmeldeverkehr'."),

    ("DDPS / VBS – Défense, protection, sport, cyber",
     "FR: défense nationale, armée, protection civile, renseignement, cybersécurité, sport, acquisitions et topographie; acteurs: DDPS, Armée suisse, SRC, OFPP, OFCS, SEPOS, armasuisse, swisstopo, OFSPO; "
     "instruments: service militaire, exercices, rapports de menace, plans d’urgence, alertes cyber, stratégie sécurité, marchés d’armement, géodonnées/cartes, programmes sportifs; "
     "vocabulaire: 'armée', 'milice', 'service militaire', 'exercice', 'mobilisation', 'renseignement', 'menace', 'espionnage', 'extrémisme', 'cyberattaque', 'protection civile', 'catastrophe', 'plan d’urgence', 'armes', 'acquisition', 'marché public', 'carte', 'géodonnées', 'Jeunesse+Sport'. "
     "DE: Landesverteidigung, Armee, Bevölkerungsschutz, Nachrichtendienst, Cybersicherheit, Sport, Beschaffung, Geodaten; Akteure: VBS, Schweizer Armee, NDB, BABS, BACS, SEPS, armasuisse, swisstopo, BASPO; "
     "Instrumente: Militärdienst, Übungen, Bedrohungsberichte, Notfallplanung, Cyberwarnungen, Sicherheitspolitik-Strategie, Rüstungsbeschaffung, Geodaten/Karten, Sportprogramme; "
     "Vokabular: 'Armee', 'Miliz', 'Militärdienst', 'Übung', 'Mobilmachung', 'Nachrichtendienst', 'Bedrohung', 'Spionage', 'Extremismus', 'Cyberangriff', 'Bevölkerungsschutz', 'Katastrophe', 'Notfallplan', 'Beschaffung', 'Ausschreibung', 'Karte', 'Geodaten', 'Jugend+Sport'."),

    ("DFF / EFD – Finances (budget, impôts, douanes, personnel, IT, achats)",
     "FR: budget, finances publiques, fiscalité/TVA, douanes/frontières, personnel fédéral, informatique, constructions/logistique, finance internationale; acteurs: DFF, AFF, AFC, OFDF, OFPER, OFIT, OFCL, SFI; "
     "instruments: budget/planification financière, perception impôts/TVA, contrôles fiscaux, contrôles douaniers, politiques RH, infrastructures IT, marchés publics/immobilier, accords/standards financiers; "
     "vocabulaire: 'budget', 'dette', 'plan financier', 'impôts', 'TVA', 'déclaration', 'contrôle fiscal', 'douanes', 'frontière', 'contrebande', 'fonction publique', 'personnel', 'informatique', 'cyber', 'marchés publics', 'bâtiments', 'transparence', 'échange automatique', 'blanchiment'. "
     "DE: Bundesfinanzen, Budget, Steuern/MWST, Zoll/Grenze, Bundespersonal, Informatik, Bauten/Logistik, internationale Finanzfragen; Akteure: EFD, EFV, ESTV, BAZG, BPA, BIT, BBL, SIF; "
     "Instrumente: Budget/Finanzplanung, Steuereinzug/MWST, Steuerkontrollen, Zollkontrollen, Personalpolitik, IT-Infrastruktur, öffentliche Beschaffung/Immobilien, Finanzabkommen/Standards; "
     "Vokabular: 'Budget', 'Schulden', 'Finanzplanung', 'Steuern', 'Mehrwertsteuer', 'Steuererklärung', 'Kontrolle', 'Zoll', 'Grenze', 'Schmuggel', 'Bundespersonal', 'Informatik', 'Cyber', 'öffentliche Beschaffung', 'Gebäude', 'Transparenz', 'AIA', 'Geldwäscherei'."),

    ("DEFR / WBF – Économie, formation, agriculture, approvisionnement, logement, service civil",
     "FR: politique économique et marché du travail, formation-recherche-innovation, agriculture, approvisionnement économique, logement, service civil; acteurs: DEFR, SECO, SEFRI, OFAG, OFAE, OFL, CIVI; "
     "instruments: rapports conjoncturels, mesures emploi/chômage, programmes formation/recherche, paiements directs, stocks obligatoires, aides au logement, affectations service civil; "
     "vocabulaire: 'chômage', 'inflation', 'croissance', 'PME', 'sanctions économiques', 'formation', 'recherche', 'innovation', 'universités', 'EPF', 'paiements directs', 'politique agricole', 'pénurie', 'approvisionnement', 'logement', 'loyers', 'coopératives', 'service civil'. "
     "DE: Wirtschaft/Arbeitsmarkt, Bildung/Forschung/Innovation, Landwirtschaft, Landesversorgung, Wohnen, Zivildienst; Akteure: WBF, SECO, SBFI, BLW, BWL, BWO, ZIVI; "
     "Instrumente: Konjunkturberichte, Arbeitsmarkt-Massnahmen, Bildungs-/Forschungsprogramme, Direktzahlungen, Pflichtlager, Wohnbauförderung, Zivildienst-Einsätze; "
     "Vokabular: 'Arbeitslosigkeit', 'Inflation', 'Wachstum', 'KMU', 'Wirtschaftssanktionen', 'Bildung', 'Forschung', 'Innovation', 'Hochschule', 'ETH', 'Direktzahlungen', 'Agrarpolitik', 'Knappheit', 'Versorgung', 'Wohnung', 'Mieten', 'Genossenschaft', 'Zivildienst'."),

    ("DETEC / UVEK – Transports, énergie, environnement, télécoms, territoire",
     "FR: transports publics/rail/aviation/routes, énergie, environnement/climat, télécommunications/médias, aménagement du territoire; acteurs: DETEC, OFT, OFAC, OFROU, OFEN, OFEV, OFCOM, ARE; "
     "instruments: concessions et financement des TP, autorisations aviation, projets routiers, politique énergétique/subventions, normes environnementales/climat, régulation télécoms/SSR/redevance, plans directeurs et planification; "
     "vocabulaire: 'CFF', 'rail', 'horaire', 'concession', 'aéroport', 'vols', 'autoroute', 'tunnel', 'chantier', 'électricité', 'réseau', 'nucléaire', 'solaire', 'éolien', 'CO2', 'biodiversité', 'pollution', 'déchets', '5G', 'internet', 'SSR', 'redevance', 'aménagement', 'territoire', 'densification'. "
     "DE: Verkehr (Bahn/Luftfahrt/Strasse), Energie, Umwelt/Klima, Telekom/Medien, Raumentwicklung; Akteure: UVEK, BAV, BAZL, ASTRA, BFE, BAFU, BAKOM, ARE; "
     "Instrumente: Konzessionen/Finanzierung ÖV, Bewilligungen Luftfahrt, Strassenprojekte, Energiepolitik/Förderungen, Umwelt- und Klimanormen, Telekomregulierung/SRG/Gebühren, Richtpläne/Planung; "
     "Vokabular: 'SBB', 'Bahn', 'Fahrplan', 'Konzession', 'Flughafen', 'Flüge', 'Autobahn', 'Tunnel', 'Baustelle', 'Strom', 'Netz', 'Kernenergie', 'Solar', 'Wind', 'CO2', 'Biodiversität', 'Verschmutzung', 'Abfall', '5G', 'Internet', 'SRG', 'Gebühr', 'Raumplanung', 'Verdichtung'."),
]




theme_labels = [t[0] for t in THEMES]
theme_texts  = [f"{t[0]}: {t[1]}" for t in THEMES]

# -------------------------------
# 8.3 Modèle d'embeddings (multilingue + léger)
# -------------------------------
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

# Embeddings des thèmes (une seule fois)
theme_emb = model.encode(
    theme_texts,
    normalize_embeddings=True,
    show_progress_bar=False
)

# -------------------------------
# 8.4 Embeddings des articles (en batch = rapide)
# -------------------------------
texts = df["text_for_theme"].fillna("").astype(str).tolist()

# Ajuste batch_size si tu vois que la RAM souffre (8GB: 32 ou 64 OK pour titres/chapeaux)
emb_articles = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

# -------------------------------
# 8.5 Similarités + décision
# -------------------------------
# Comme tout est normalisé, le produit scalaire = cosine similarity
scores = emb_articles @ theme_emb.T  # shape: (n_articles, n_themes)

best_idx = scores.argmax(axis=1)
best_score = scores.max(axis=1)

df["main_theme"] = [theme_labels[i] for i in best_idx]
df["theme_score"] = best_score.astype(float)

# Option: seuil pour classer en "Autre" quand c'est trop incertain
THRESHOLD = 0.25
df.loc[df["theme_score"] < THRESHOLD, "main_theme"] = "Autre"

print("✅ Terminé. Répartition des thèmes :")
print(df["main_theme"].value_counts().head(30))

df[["id", "pubtime", "language", "medium_name", "head", "subhead", "main_theme", "theme_score"]].head(10)


In [ ]:
import pandas as pd

# 1) Préparer month
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")
df["month"] = df["pubtime"].dt.to_period("M").astype(str)

# 2) Counts par mois x thème
counts = (
    df.dropna(subset=["month", "main_theme"])
      .groupby(["month", "main_theme"])
      .size()
      .reset_index(name="n")
)

# 3) Total annuel par thème (somme sur tous les mois)
totals = (
    counts.groupby("main_theme")["n"]
          .sum()
          .reset_index(name="total_theme_year")
)

# 4) Pourcentage du mois par rapport au total annuel du thème
pct_long = counts.merge(totals, on="main_theme", how="left")
pct_long["pct_of_theme_year"] = 100 * pct_long["n"] / pct_long["total_theme_year"]

# (optionnel) arrondi
pct_long["pct_of_theme_year"] = pct_long["pct_of_theme_year"].round(2)

pct_long.sort_values(["main_theme", "month"]).head(20)

pct_wide = (
    pct_long.pivot(index="month", columns="main_theme", values="pct_of_theme_year")
            .fillna(0)
            .sort_index()
)

pct_wide.head()


In [ ]:
import pandas as pd

# pubtime -> datetime (si c'est déjà un datetime/date, ça ne casse rien)
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")

# mois au format YYYY-MM
df["month"] = df["pubtime"].dt.to_period("M").astype(str)

themes_by_month_long = (
    df.dropna(subset=["month", "main_theme"])
      .groupby(["month", "main_theme"])
      .size()
      .reset_index(name="n")
      .sort_values(["month", "n"], ascending=[True, False])
)

themes_by_month_long.head(20)

themes_by_month_wide = (
    themes_by_month_long
      .pivot(index="month", columns="main_theme", values="n")
      .fillna(0)
      .astype(int)
      .sort_index()
)

themes_by_month_wide.head()
